# Pregunta 3 — CFA, SEM y comparación por género

**Métodos Cuantitativos de Investigación en Negocios — Tarea 2 (2026)**

## Enunciado de la pregunta 3

**3.** Como recién llegado a la consultora antes mencionada, le han incluido en un segundo proyecto, relacionado con los efectos de las condiciones de trabajo sobre la satisfacción y permanencia de los trabajadores. En este contexto, se han considerado los siguientes constructos relevantes para la investigación:

- **Actitud hacia compañeros de trabajo:** Actitud del empleado hacia/con sus compañeros de trabajo con los que interactúa regularmente.
- **Percepción del ambiente de trabajo:** Percepción del trabajador respecto a su lugar de trabajo.
- **Satisfacción con el trabajo:** Medición del nivel de satisfacción del empleado con su trabajo.
- **Compromiso organizacional:** Grado en que el empleado se identifica y siente parte de la empresa.
- **Intención de permanecer en el trabajo:** Grado en el que el trabajador pretende continuar trabajando para la empresa.
- **Características personales:** Edad, Género, y experiencia en años del trabajador.

Cada uno de estos constructos fue medido en escalas de múltiples ítems que se describen a continuación, y que fueron aplicadas en un cuestionario a 400 empleados de diversas empresas del sector de telecomunicaciones, disponibles en el archivo `Laboral.sav`.

![Tabla de constructos e ítems de la pregunta 3](../assets/p3_tabla_constructos.png)

Las escalas específicas de cada ítem se encuentran detalladas en el archivo de datos `Laboral.sav`.

Gracias a sus conocimientos en el área de comportamiento organizacional, usted ha identificado dos posibles modelos que expliquen la relación entre las condiciones de trabajo percibidas por el trabajador, y su satisfacción e intenciones de permanecer en la empresa. Estos modelos difieren principalmente en si el efecto de la precepción de ambiente de trabajo y la actitud hacia los compañeros de trabajo tienen efectos completamente mediados o parcialmente mediados sobre la intención de permanecer en la empresa.

Cada uno de los modelos se describe a continuación:

![Modelo completamente mediado](../assets/p3_modelo_completamente_mediado.png)

![Modelo parcialmente mediado](../assets/p3_modelo_parcialmente_mediado.png)

Su labor consiste en determinar:

**a)** Qué tan bueno es el modelo de medición, incluyendo medidas de confiabilidad y validez de escalas.  
**b)** Determinar cuál de los dos modelos representa de mejor forma los datos recopilados.  
**c)** Para el modelo identificado como superior, evalúe si las relaciones entre las distintas variables se comportan de la misma forma entre hombres y mujeres.  
**d)** Explique sus resultados y conclusiones.

## Respuesta

La pregunta se resuelve en dos etapas: primero se evalúa el modelo de medición mediante CFA, confiabilidad y validez; luego se comparan los modelos estructurales completamente mediado y parcialmente mediado mediante SEM. Finalmente, el modelo superior se analiza por género para evaluar si las relaciones se comportan de forma similar entre hombres y mujeres.


In [1]:
%pip install semopy


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import pyreadstat
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)
# Carpeta de datos.
# El repositorio deja las bases SPSS en Tarea2/data para que todos los notebooks sean reproducibles.
# La lógica revisa varias ubicaciones habituales: ejecución desde Tarea2/notebooks, desde Tarea2, desde la raíz del repo o desde Colab.
candidatas = [
    Path.cwd() / 'data',
    Path.cwd().parent / 'data',
    Path.cwd() / 'Tarea2' / 'data',
    Path.cwd()  # respaldo: bases junto al notebook (por ejemplo, en Colab tras subirlas)
]
DATA_DIR = next((ruta for ruta in candidatas if (ruta / 'Med_Mod.sav').exists()), candidatas[0])

print('Directorio de datos:', DATA_DIR)

Directorio de datos: /Users/sbc/projects/Metodos-Cuantitativos-DEN/Tarea2/data


## 3.a) Modelo de medición

Antes de comparar los modelos estructurales, se debe verificar que los constructos estén bien medidos. Esta etapa corresponde a un **Análisis Factorial Confirmatorio (CFA)**. La idea es confirmar que cada grupo de ítems mide el constructo teórico que corresponde.

En este problema hay cinco constructos latentes:

- $AC$: actitud hacia compañeros de trabajo.
- $PA$: percepción del ambiente de trabajo.
- $ST$: satisfacción con el trabajo.
- $CO$: compromiso organizacional.
- $IP$: intención de permanecer en el trabajo.

Cada constructo se mide con cuatro indicadores observados. La forma general de una ecuación de medición es:

$$x_{ij}=\lambda_{ij}\eta_j+\varepsilon_{ij}$$

donde $x_{ij}$ es el ítem observado, $\eta_j$ es el constructo latente, $\lambda_{ij}$ es la carga factorial y $\varepsilon_{ij}$ es el error de medición del ítem.

Aplicado a esta tarea, el modelo de medición se escribe así:

$$AC =~ AC1 + AC2 + AC3 + AC4$$
$$PA =~ PA1 + PA2 + PA3 + PA4$$
$$ST =~ ST1 + ST2 + ST3 + ST4$$
$$CO =~ CO1 + CO2 + CO3 + CO4$$
$$IP =~ IP1 + IP2 + IP3 + IP4$$

En esta etapa se revisa:

1. **Ajuste global del modelo de medición:** si la estructura de cinco factores reproduce razonablemente la matriz de covarianzas observada.
2. **Cargas factoriales:** si los ítems se asocian fuertemente con su constructo.
3. **Confiabilidad:** si los ítems de cada constructo son consistentes entre sí.
4. **Validez convergente:** si los indicadores de un mismo constructo comparten suficiente varianza.
5. **Validez discriminante:** si los constructos son distintos entre sí y no están midiendo exactamente lo mismo.

Este paso es necesario porque no tendría sentido comparar relaciones causales entre constructos si antes no se verifica que esos constructos están medidos de forma razonable.


In [3]:
from semopy import Model, calc_stats
df,meta=pyreadstat.read_sav(DATA_DIR/'Laboral.sav',apply_value_formats=False)
constructos={'AC':['AC1','AC2','AC3','AC4'],'PA':['PA1','PA2','PA3','PA4'],'ST':['ST1','ST2','ST3','ST4'],'CO':['CO1','CO2','CO3','CO4'],'IP':['IP1','IP2','IP3','IP4']}
obs=sum(constructos.values(),[]); datos=df[obs+['GENERO']].dropna()
print('Casos:',len(datos),'Hombres:',sum(datos.GENERO==0),'Mujeres:',sum(datos.GENERO==1))

Casos: 400 Hombres: 200 Mujeres: 200


In [4]:
# Estimación del CFA puro: cinco factores libremente correlacionados.
# Esta celda responde cuantitativamente si cada ítem carga en el constructo que corresponde.
medicion = '\n'.join(f"{k} =~ {' + '.join(v)}" for k, v in constructos.items())
latentes = list(constructos)
covarianzas = []
for i, a in enumerate(latentes):
    for b in latentes[i+1:]:
        covarianzas.append(f'{a} ~~ {b}')

cfa_desc = medicion + '\n' + '\n'.join(covarianzas)
modelo_cfa = Model(cfa_desc)
modelo_cfa.fit(datos)
ajuste_cfa = calc_stats(modelo_cfa).iloc[0]
param_cfa = modelo_cfa.inspect(std_est=True)

cargas_cfa = param_cfa[(param_cfa.op == '~') & (param_cfa.rval.isin(constructos.keys()))][
    ['rval', 'lval', 'Estimate', 'Est. Std', 'Std. Err', 'z-value', 'p-value']
].copy()
cargas_cfa.columns = ['Constructo', 'Ítem', 'Carga no est.', 'Carga est.', 'EE', 'z', 'p']
cargas_cfa['orden'] = cargas_cfa['Constructo'].map({k: i for i, k in enumerate(constructos)}) * 10 + cargas_cfa['Ítem'].str.extract(r'(\d+)').astype(int)[0]
cargas_cfa = cargas_cfa.sort_values('orden').drop(columns='orden')

print('Ajuste global del CFA de medición')
print(pd.Series({
    'chi2': ajuste_cfa.chi2,
    'gl': ajuste_cfa.DoF,
    'CFI': ajuste_cfa.CFI,
    'TLI': ajuste_cfa.TLI,
    'RMSEA': ajuste_cfa.RMSEA
}).round(4).to_string())

print('\nCargas factoriales estandarizadas por ítem')
print(cargas_cfa.round(4).to_string(index=False))


Ajuste global del CFA de medición
chi2     220.3091
gl       160.0000
CFI        0.9850
TLI        0.9822
RMSEA      0.0307

Cargas factoriales estandarizadas por ítem
Constructo Ítem  Carga no est.  Carga est.        EE          z    p
        AC  AC1         1.0000      0.8219         -          -    -
        AC  AC2         1.2363      0.8203   0.06723  18.388501  0.0
        AC  AC3         1.0375      0.8372  0.054989  18.866634  0.0
        AC  AC4         1.1467      0.8155  0.062832  18.250812  0.0
        PA  PA1         1.0000      0.6917         -          -    -
        PA  PA2         1.0336      0.8040  0.073363  14.089158  0.0
        PA  PA3         0.8204      0.7783  0.059766  13.726961  0.0
        PA  PA4         0.9137      0.8229  0.063741  14.335176  0.0
        ST  ST1         1.0000      0.7371         -          -    -
        ST  ST2         1.0226      0.7368  0.081086  12.611086  0.0
        ST  ST3         0.9294      0.6968  0.076859  12.092472  0.0
    

### Confiabilidad, validez convergente y validez discriminante

Después de estimar el modelo de medición, se evalúa la calidad de cada escala.

### Confiabilidad

La confiabilidad indica si los ítems de un constructo son consistentes entre sí. Se reportan dos medidas:

- **Alfa de Cronbach:** mide consistencia interna bajo el supuesto de que los ítems son relativamente equivalentes.
- **Confiabilidad compuesta (CR):** usa las cargas factoriales del CFA, por lo que es más coherente con modelos de ecuaciones estructurales.

Como regla práctica, valores de alfa y CR iguales o superiores a 0.70 se consideran adecuados.

### Validez convergente

La validez convergente evalúa si los ítems que deberían medir un mismo constructo efectivamente comparten suficiente varianza. Se usa la **varianza media extraída (AVE)**:

$$AVE = \frac{\sum \lambda_i^2}{k}$$

Un AVE mayor o igual a 0.50 indica que el constructo explica al menos la mitad de la varianza de sus indicadores.

### Validez discriminante

La validez discriminante evalúa si los constructos son distintos entre sí. Se revisa con:

- **Correlaciones latentes:** no deberían ser excesivamente altas.
- **Fornell–Larcker:** la raíz cuadrada del AVE de cada constructo debe superar sus correlaciones con los demás constructos; así cada constructo comparte más varianza con sus propios ítems que con los otros conceptos.
- **HTMT:** valores menores a 0.85 suelen considerarse evidencia de discriminación adecuada.

Si una escala tiene buena confiabilidad, AVE suficiente y discriminación adecuada, se puede usar con más confianza en el modelo estructural.


A modo de ilustración del cálculo de CR y AVE, para $AC$ las cargas estandarizadas del CFA son $\lambda=(0.8219,\,0.8203,\,0.8372,\,0.8155)$, con $\sum\lambda_i=3.2949$ y $\sum(1-\lambda_i^2)=1.2857$. Reemplazando:

$$CR=\frac{(3.2949)^2}{(3.2949)^2+1.2857}=\frac{10.856}{12.142}=0.894,\qquad AVE=\frac{\sum\lambda_i^2}{4}=0.679.$$

In [5]:
def alpha(d):
 k=d.shape[1]; return k/(k-1)*(1-d.var(ddof=1).sum()/d.sum(axis=1).var(ddof=1))
filas=[]
for c,its in constructos.items():
 r=cargas_cfa[cargas_cfa['Constructo'].eq(c)].set_index('Ítem').reindex(its)['Carga est.'].astype(float)
 cr=r.sum()**2/(r.sum()**2+(1-r**2).sum()); ave=(r**2).mean()
 filas.append([c,alpha(datos[its]),cr,ave,np.sqrt(ave),r.min(),r.max()])
calidad=pd.DataFrame(filas,columns=['Constructo','Alfa','CR','AVE','raíz AVE','Carga mínima','Carga máxima'])
print(calidad.round(4).to_string(index=False))

# Correlaciones latentes implicadas por el propio CFA (estandarizadas).
# Se usan estas, y no puntajes factoriales estimados, porque son las que el modelo reproduce.
lat=param_cfa[(param_cfa.op=='~~')&param_cfa.lval.isin(constructos)&param_cfa.rval.isin(constructos)&(param_cfa.lval!=param_cfa.rval)]
print('\nCorrelaciones latentes implicadas por el CFA:')
for _,r in lat.iterrows(): print(f"{r.lval}-{r.rval}: {r['Est. Std']:.3f}")
max_corr=lat['Est. Std'].abs().max(); min_rave=calidad['raíz AVE'].min()
print(f'\nFornell–Larcker: mínima raíz de AVE = {min_rave:.3f} > correlación latente máxima = {max_corr:.3f} -> se cumple')

def htmt(a,b):
 A,B=constructos[a],constructos[b]; R=datos[A+B].corr().abs(); h=R.loc[A,B].values.mean(); ma=R.loc[A,A].values[np.triu_indices(4,1)].mean(); mb=R.loc[B,B].values[np.triu_indices(4,1)].mean(); return h/np.sqrt(ma*mb)
print('\nHTMT:')
for i,a in enumerate(constructos):
 for b in list(constructos)[i+1:]: print(f'{a}-{b}: {htmt(a,b):.3f}')

Constructo   Alfa     CR    AVE  raíz AVE  Carga mínima  Carga máxima
        AC 0.8908 0.8941 0.6786    0.8238        0.8155        0.8372
        PA 0.8474 0.8576 0.6019    0.7758        0.6917        0.8229
        ST 0.8112 0.8115 0.5185    0.7201        0.6968        0.7371
        CO 0.8227 0.8342 0.5638    0.7509        0.5836        0.8854
        IP 0.8863 0.8900 0.6698    0.8184        0.7410        0.8641

Correlaciones latentes implicadas por el CFA:
AC-PA: 0.257
AC-ST: 0.025
AC-CO: 0.307
AC-IP: 0.308
PA-ST: 0.228
PA-CO: 0.497
PA-IP: 0.562
ST-CO: 0.190
ST-IP: 0.214
CO-IP: 0.553

Fornell–Larcker: mínima raíz de AVE = 0.720 > correlación latente máxima = 0.562 -> se cumple

HTMT:
AC-PA: 0.260
AC-ST: 0.044
AC-CO: 0.275
AC-IP: 0.310
PA-ST: 0.231
PA-CO: 0.495
PA-IP: 0.569
ST-CO: 0.193
ST-IP: 0.217
CO-IP: 0.501


### Respuesta cuantitativa del modelo de medición

El modelo de medición presenta buen ajuste global. El CFA de cinco factores correlacionados obtuvo:

$$\chi^2(160)=220.3091,\quad CFI=0.9850,\quad TLI=0.9822,\quad RMSEA=0.0307$$

Estos valores son favorables: CFI y TLI están sobre 0.95 y RMSEA está bajo 0.06. Por lo tanto, la estructura de medición propuesta reproduce adecuadamente los datos observados.

La tabla de cargas factoriales muestra cuánto se asocia cada ítem con su constructo latente. Las cargas estandarizadas van desde 0.5836 hasta 0.8854. La mayoría de los ítems supera 0.70. Los ítems más bajos son CO1 = 0.5836, CO3 = 0.6571, PA1 = 0.6917 y ST3 = 0.6968, pero ninguno cae a un nivel problemático extremo y todos pertenecen a escalas con confiabilidad compuesta adecuada.

La calidad agregada por constructo es la siguiente:

| Constructo | Alfa | CR | AVE | Carga mínima | Carga máxima |
|---|---:|---:|---:|---:|---:|
| AC | 0.8908 | 0.8941 | 0.6786 | 0.8155 | 0.8372 |
| PA | 0.8474 | 0.8576 | 0.6019 | 0.6917 | 0.8229 |
| ST | 0.8112 | 0.8115 | 0.5185 | 0.6968 | 0.7371 |
| CO | 0.8227 | 0.8341 | 0.5638 | 0.5836 | 0.8854 |
| IP | 0.8863 | 0.8900 | 0.6698 | 0.7410 | 0.8641 |

La confiabilidad es adecuada porque todos los alfa de Cronbach son superiores a 0.80 y todas las confiabilidades compuestas son superiores a 0.81. La validez convergente también es adecuada porque todos los AVE son mayores a 0.50.

La validez discriminante es satisfactoria. El HTMT máximo es 0.569 para PA--IP, muy por debajo del criterio de 0.85. Además, la mayor correlación latente observada es 0.607 entre PA e IP, lo que indica asociación, pero no redundancia entre constructos.

En conclusión, el modelo de medición es bueno: presenta ajuste global adecuado, cargas factoriales mayoritariamente altas, confiabilidad suficiente, validez convergente y validez discriminante. Por eso es metodológicamente razonable avanzar a la comparación de los modelos estructurales completo y parcial.

### Conclusión 3.a

El modelo de medición es adecuado. Cuantitativamente, el ajuste global del CFA es bueno porque $CFI=0.9850$, $TLI=0.9822$ y $RMSEA=0.0307$. Además, todas las escalas presentan confiabilidad suficiente, con alfa entre 0.8112 y 0.8908, y confiabilidad compuesta entre 0.8115 y 0.8941.

La validez convergente también se cumple porque todos los AVE son mayores a 0.50. La validez discriminante es aceptable porque el HTMT máximo es 0.569, inferior al criterio de 0.85. Aunque algunos ítems tienen cargas algo más bajas, como CO1 = 0.5836, todas las escalas mantienen indicadores globales adecuados.

Por lo tanto, se concluye que el modelo de medición es suficientemente bueno para continuar con la comparación de los modelos estructurales completamente mediado y parcialmente mediado.



## 3.b) Comparación de los modelos estructurales

Una vez validado el modelo de medición, se comparan los dos modelos estructurales que aparecen en las figuras del enunciado.

### Modelo completamente mediado

El modelo completamente mediado supone que la percepción del ambiente de trabajo ($PA$) y la actitud hacia compañeros ($AC$) no afectan directamente la intención de permanecer ($IP$). Sin embargo, según la figura original, $PA$ y $AC$ sí predicen tanto satisfacción ($ST$) como compromiso organizacional ($CO$), y además $ST$ predice tanto $CO$ como $IP$.

Sus ecuaciones estructurales son:

$$ST = \gamma_1 PA + \gamma_2 AC + \zeta_{ST}$$
$$CO = \gamma_3 PA + \gamma_4 AC + eta_1 ST + \zeta_{CO}$$
$$IP = eta_2 ST + eta_3 CO + \zeta_{IP}$$

La lógica es: las condiciones de trabajo inciden en satisfacción y compromiso; satisfacción incide en compromiso e intención de permanencia; y compromiso incide en permanencia. No hay caminos directos desde $PA$ o $AC$ hacia $IP$.

### Modelo parcialmente mediado

El modelo parcialmente mediado mantiene los caminos anteriores, pero agrega efectos directos desde $PA$ y $AC$ hacia $IP$. Esto permite que el ambiente laboral y la relación con compañeros influyan en la intención de permanecer no solo a través de satisfacción y compromiso, sino también directamente.

Sus ecuaciones estructurales son:

$$ST = \gamma_1 PA + \gamma_2 AC + \zeta_{ST}$$
$$CO = \gamma_3 PA + \gamma_4 AC + eta_1 ST + \zeta_{CO}$$
$$IP = \gamma_5 PA + \gamma_6 AC + eta_2 ST + eta_3 CO + \zeta_{IP}$$

La diferencia clave está en la tercera ecuación: el modelo parcial agrega $PA ightarrow IP$ y $AC ightarrow IP$.

### Cómo se decide cuál modelo es mejor

Los modelos son **anidados**, porque el modelo completamente mediado es una versión restringida del parcialmente mediado. Por eso se puede comparar la pérdida de ajuste al imponer las restricciones $PAightarrow IP=0$ y $ACightarrow IP=0$.

Se revisan dos tipos de evidencia: índices de ajuste global (CFI, TLI, RMSEA, AIC y BIC) y diferencia de chi-cuadrado. Si $\Delta\chi^2$ es significativa, el modelo parcial mejora el ajuste de forma estadísticamente relevante.

In [6]:
med='\n'.join(f"{k} =~ {' + '.join(v)}" for k,v in constructos.items())
# Especificación corregida según las figuras del enunciado:
# Completo: PA y AC predicen ST y CO; ST predice CO e IP; CO predice IP.
# Parcial: agrega caminos directos PA->IP y AC->IP.
completo=med+'\nST ~ PA + AC\nCO ~ PA + AC + ST\nIP ~ ST + CO'
parcial=med+'\nST ~ PA + AC\nCO ~ PA + AC + ST\nIP ~ PA + AC + ST + CO'
def ajustar(desc,d):
    m=Model(desc); m.fit(d); return m,calc_stats(m).iloc[0],m.inspect(std_est=True)
mc,fc,pc=ajustar(completo,datos); mp,fp,pp=ajustar(parcial,datos)

# AIC y BIC calculados sobre el estadístico chi2: AIC = chi2 + 2k, BIC = chi2 + k·ln(n).
n=len(datos)
def criterios(m,f):
    k=len(m.param_vals); return k, f.chi2+2*k, f.chi2+k*np.log(n)
kc,aic_c,bic_c=criterios(mc,fc); kp,aic_p,bic_p=criterios(mp,fp)

tabla=pd.DataFrame({'Completo':[fc.chi2,fc.DoF,fc.CFI,fc.TLI,fc.RMSEA,kc,aic_c,bic_c],
                    'Parcial':[fp.chi2,fp.DoF,fp.CFI,fp.TLI,fp.RMSEA,kp,aic_p,bic_p]},
                   index=['chi2','gl','CFI','TLI','RMSEA','k (parámetros)','AIC (chi2+2k)','BIC (chi2+k·ln n)'])
print(tabla.round(4).to_string())

dchi=fc.chi2-fp.chi2; ddf=fc.DoF-fp.DoF
print(f'\nDiferencia chi-cuadrado = {fc.chi2:.4f} - {fp.chi2:.4f} = {dchi:.4f}, gl = {ddf:.0f}, p = {stats.chi2.sf(dchi,ddf):.4g}')
print(f'chi2 crítico (95%, gl={ddf:.0f}) = {stats.chi2.ppf(.95,ddf):.4f}')

                   Completo   Parcial
chi2               267.1340  220.3087
gl                 162.0000  160.0000
CFI                  0.9738    0.9850
TLI                  0.9693    0.9822
RMSEA                0.0403    0.0307
k (parámetros)      48.0000   50.0000
AIC (chi2+2k)      363.1340  320.3087
BIC (chi2+k·ln n)  554.7243  519.8819

Diferencia chi-cuadrado = 267.1340 - 220.3087 = 46.8253, gl = 2, p = 6.792e-11
chi2 crítico (95%, gl=2) = 5.9915


### Relaciones estructurales del modelo seleccionado

Una vez elegido el modelo con mejor ajuste, se interpretan sus coeficientes estructurales. En la especificación correcta del modelo parcial se consideran nueve caminos:

- $PA ightarrow ST$ y $AC ightarrow ST$: efectos de condiciones laborales sobre satisfacción.
- $PA ightarrow CO$, $AC ightarrow CO$ y $ST ightarrow CO$: efectos sobre compromiso organizacional.
- $PA ightarrow IP$, $AC ightarrow IP$, $ST ightarrow IP$ y $CO ightarrow IP$: efectos directos sobre intención de permanecer.

Para cada camino se observa el coeficiente, su error estándar, el estadístico $z$ y el valor-p. Si $p<0.05$, se concluye que la relación es estadísticamente significativa al 95% de confianza.

In [7]:
paths=pp[(pp.op=='~')&pp.lval.isin(['ST','CO','IP'])][['lval','rval','Estimate','Est. Std','Std. Err','z-value','p-value']]
print(paths.round(4).to_string(index=False))

lval rval  Estimate  Est. Std  Std. Err   z-value   p-value
  ST   PA    0.1848    0.2373  0.049428  3.739257  0.000185
  ST   AC   -0.0306   -0.0356  0.051809 -0.591183  0.554398
  CO   PA    0.4969    0.4272  0.077823  6.384778       0.0
  CO   AC    0.2505    0.1948  0.069761  3.590861   0.00033
  CO   ST    0.1306    0.0875  0.081605  1.600225  0.109549
  IP   PA    0.1975    0.3540  0.033796  5.843721       0.0
  IP   AC    0.0709    0.1149  0.029954    2.3667  0.017947
  IP   ST    0.0485    0.0678  0.035301  1.374472  0.169295
  IP   CO    0.1576    0.3285   0.02974  5.297898       0.0


## 3.c) Comparación de relaciones entre hombres y mujeres

El enunciado pide evaluar si, para el modelo superior, las relaciones se comportan igual entre hombres y mujeres. Para responderlo, se estima el modelo parcial por separado en ambos grupos y se comparan los coeficientes estructurales.

La comparación se realiza con:

$$z=\frac{b_H-b_M}{\sqrt{SE_H^2+SE_M^2}}$$

donde $b_H$ es el coeficiente estimado en hombres, $b_M$ es el coeficiente estimado en mujeres, y $SE_H$ y $SE_M$ son sus errores estándar.

La hipótesis nula de cada comparación es:

$$H_0: b_H=b_M$$

La hipótesis alternativa es:

$$H_1: b_H\neq b_M$$

Si $p<0.05$, se concluye que esa relación estructural difiere significativamente entre hombres y mujeres. Esta comparación permite identificar si el modelo funciona igual para ambos grupos o si algunas relaciones cambian de magnitud según género.

Como nota metodológica, una evaluación multigrupo completamente confirmatoria también podría incluir pruebas de invariancia configural, métrica y escalar. En esta tarea, la comparación se concentra en las relaciones estructurales del modelo seleccionado.


In [8]:
gr={}; stats_gr={}
for g,nombre in [(0,'Hombres'),(1,'Mujeres')]:
    m,f,p=ajustar(parcial,datos[datos.GENERO==g]); gr[g]=p[(p.op=='~')&p.lval.isin(['ST','CO','IP'])]; stats_gr[g]=f
    print(nombre,'n=',sum(datos.GENERO==g),'CFI=',round(f.CFI,3),'RMSEA=',round(f.RMSEA,3))
fil=[]
for lhs,rhs in [('ST','PA'),('ST','AC'),('CO','PA'),('CO','AC'),('CO','ST'),('IP','PA'),('IP','AC'),('IP','ST'),('IP','CO')]:
    a=gr[0][(gr[0].lval==lhs)&(gr[0].rval==rhs)].iloc[0]; b=gr[1][(gr[1].lval==lhs)&(gr[1].rval==rhs)].iloc[0]
    z=(a.Estimate-b.Estimate)/np.sqrt(float(a['Std. Err'])**2+float(b['Std. Err'])**2)
    fil.append([f'{rhs}→{lhs}',a.Estimate,b.Estimate,z,2*stats.norm.sf(abs(z))])
print(pd.DataFrame(fil,columns=['Relación','Hombres','Mujeres','z diferencia','p']).round(4).to_string(index=False))

Hombres n= 200 CFI= 0.981 RMSEA= 0.031
Mujeres n= 200 CFI= 0.972 RMSEA= 0.041
Relación  Hombres  Mujeres  z diferencia      p
   PA→ST   0.2594   0.1443        1.0590 0.2896
   AC→ST  -0.2145   0.3509       -3.3134 0.0009
   PA→CO   0.6348   0.5099        0.7431 0.4574
   AC→CO   0.3620   1.0379       -2.4184 0.0156
   ST→CO   0.0729   0.0830       -0.0587 0.9532
   PA→IP   0.0984   0.2385       -1.9668 0.0492
   AC→IP  -0.0045   0.0038       -0.0724 0.9423
   ST→IP   0.0601   0.0171        0.5756 0.5649
   CO→IP   0.1327   0.2050       -1.1515 0.2495


### Estadístico de comparación y sustitución (ejemplo AC→ST)

El estadístico de comparación bilateral al 95% de confianza es $z_c=\pm 1.9600$. A modo de ilustración, para $ACightarrow ST$ los coeficientes no estandarizados fueron $b_H=-0.2145$ y $b_M=0.3509$. Con sus errores estándar, el contraste entrega:

$$z=-3.3134.$$

Como $|-3.3134|>1.9600$ y $p=0.0009$, se rechaza $H_0: b_H=b_M$ en esa relación: la pendiente es negativa entre hombres y positiva entre mujeres. Aplicando la misma regla fila a fila, también difieren $ACightarrow CO$ ($p=0.0156$) y $PAightarrow IP$ ($p=0.0492$). Las demás relaciones no difieren al 5%.

Respecto de la invariancia métrica, este notebook reporta el ajuste configural por grupo y compara caminos estructurales como aproximación informativa. Una prueba formal de invariancia métrica requiere estimar un modelo multigrupo con cargas factoriales igualadas simultáneamente entre grupos; `semopy` permite estimaciones por grupo, pero no impone esas restricciones multigrupo de forma directa en esta especificación. Por ello, la conclusión sobre diferencias por género debe interpretarse como evidencia estructural exploratoria, no como una prueba confirmatoria completa de invariancia.

## 3.d) Explicación y conclusión

El modelo de medición es sólido: el CFA puro de cinco factores correlacionados ajusta muy bien ($\chi^2(160)=220.31$, CFI=0.985, TLI=0.982, RMSEA=0.031) y todas las escalas muestran confiabilidad adecuada (alfa 0.811–0.891; CR 0.811–0.894), AVE superior a 0.50 y discriminación adecuada: la mínima raíz de AVE (0.720) supera la correlación latente máxima (0.562) y el HTMT máximo es 0.569, muy por debajo de 0.85.

El modelo parcial presenta mejor ajuste (CFI=0.985, TLI=0.982, RMSEA=0.031) que el completo (CFI=0.974, TLI=0.969, RMSEA=0.040), con mejora significativa, $\Delta\chi^2(2)=46.83$, $p<0.001$. Con AIC y BIC calculados sobre $\chi^2$, ambos criterios también favorecen al parcial (AIC 320.31 contra 363.13; BIC 519.88 contra 554.72). Por ello se selecciona el **modelo parcialmente mediado**.

En el modelo parcial, PA→ST, PA→CO, AC→CO, PA→IP, AC→IP y CO→IP son significativos. AC→ST, ST→CO y ST→IP no son significativos al 5%. Por género, difieren AC→ST ($p=0.0009$), AC→CO ($p=0.0156$) y PA→IP ($p=0.0492$); las demás relaciones no difieren al 5%. Por tanto, **el modelo no se comporta completamente igual entre hombres y mujeres**, aunque la comparación debe interpretarse con cautela porque no se estimó una prueba formal completa de invariancia métrica.